# 第15回(最終回)総まとめ — この15回で、何が言えるようになったか

**この授業の集大成です。**

1つのモデル(正解率 84.6%)を用意しました。
この**1つの数字**を、皆さんが学んだ**5つの視点**で分解します。

1. 視点1(第3回)正解率だけでは足りない
2. 視点2(第5回)学習しすぎると悪くなる
3. 視点3(第6回)見せていない変化に耐えるか
4. 視点4(第13回)自信は較正されているか
5. 視点5(第14回)属性ごとに見るとどうか

**先に結論**:5つのうち **3つで問題が見つかり、2つでは見つかりませんでした。**

所要時間:実習 約25分 / 最終課題 約90分

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

このノートブックは、**この授業の総まとめ**です。
1つのモデルを用意し、皆さんが学んだ5つの視点で「1つの数字」を分解します。

In [ ]:
# 準備(このセルを最初に1回だけ実行。初回は1〜2分かかります)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, japanize_matplotlib
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
plt.rcParams["font.size"] = 12

元 = fetch_openml("adult", version=2, as_frame=True); df = 元.frame.dropna()
y = (df["class"] == ">50K").astype(int)
性別 = (df["sex"] == "Male").astype(int)
使う項目 = ["age","workclass","education-num","marital-status","occupation",
            "relationship","hours-per-week","capital-gain","capital-loss"]
X = df[使う項目].copy()
文字列 = X.select_dtypes(include=["category","object"]).columns
X[文字列] = OrdinalEncoder().fit_transform(X[文字列].astype(str))
Xtr, Xte, ytr, yte, 性tr, 性te = train_test_split(X, y, 性別, test_size=0.3, random_state=0, stratify=y)

モデル = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(Xtr, ytr)
確率 = モデル.predict_proba(Xte)[:, 1]
予測 = (確率 >= 0.5).astype(int)

print("■ 見出しの数字 ──────────────────────")
print(f"    正解率 {(予測==yte).mean()*100:.1f}%")
print("──────────────────────────────────")
print("この数字だけを聞いて、このモデルを採用してよいでしょうか。")
print("5つの視点で分解します。")

## 視点1(第3回)正解率だけでは足りない

「精度」と言われたら、次に聞くこと ── **見逃しと空振り、それぞれ何件でしたか**。

そしてもう1つ。**何と比べた数字ですか**(第12回)。

In [ ]:
# ===== 視点1(第3回)正解率だけでは足りない =====
tp = ((予測==1) & (yte==1)).sum()
fp = ((予測==1) & (yte==0)).sum()
fn = ((予測==0) & (yte==1)).sum()
tn = ((予測==0) & (yte==0)).sum()

print("混同行列")
print(f"            実際:高収入   実際:低収入")
print(f"  判定:高収入 {tp:>8,}     {fp:>8,}")
print(f"  判定:低収入 {fn:>8,}     {tn:>8,}")
print()
print(f"適合率(高収入と判定して当たる率) {tp/(tp+fp)*100:.1f}%")
print(f"再現率(高収入の人を見つける率)   {tp/(tp+fn)*100:.1f}%")
print()
print(f"【比較】「全員を低収入」と答えるだけのモデルの正解率: {(1-yte.mean())*100:.1f}%")
print(f"        このモデルとの差: {(予測==yte).mean()*100-(1-yte.mean())*100:.1f} ポイント")
print()
print("高収入の人の4割近くを見逃しています。")
print("そして「何も学習しないモデル」との差は、思ったより小さいはずです。")
print("→ 84.6% という数字は、比べて初めて意味を持ちます。")

## 視点2(第5回)学習しすぎると悪くなる

損失が下がり続けている ── それは「順調」ではなく、**「性能が落ちている最中」**かもしれません。

木の深さを変えて、訓練データとテストデータの成績がどう動くかを見ます。

In [ ]:
# ===== 視点2(第5回)学習しすぎると悪くなる =====
print(f"{'木の深さ':<10}{'訓練':>10}{'テスト':>10}{'葉の数':>10}")
print("-" * 42)
記録 = []
for d in [3, 5, 8, 15, 25, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    名 = "制限なし" if d is None else f"深さ{d}"
    記録.append((名, t.score(Xtr,ytr)*100, t.score(Xte,yte)*100, t.get_n_leaves()))
    print(f"{名:<10}{t.score(Xtr,ytr)*100:>9.1f}%{t.score(Xte,yte)*100:>9.1f}%{t.get_n_leaves():>10,}")

plt.figure(figsize=(7.5, 4))
x = range(len(記録))
plt.plot(x, [r[1] for r in 記録], "o-", label="訓練データ", color="#3A46A8")
plt.plot(x, [r[2] for r in 記録], "o-", label="テストデータ", color="#E08A2B")
plt.xticks(x, [r[0] for r in 記録]); plt.ylabel("正解率(%)")
plt.title("深くするほど良くなる、わけではない")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("訓練の数字だけを見せられていたら、どれを選んでいましたか?")

## 視点3(第6回)見せていない変化に耐えるか

モデルが強いのは、**見せたものに対してだけ**です。
入力に少しばらつきを足したら、成績はどうなるでしょうか。

**予想してから実行してください。**

In [ ]:
# ===== 視点3(第6回)見せていない変化に耐えるか =====
print(f"{'週労働時間を揺らす':<22}{'正解率':>10}")
print("-" * 34)
for 割合 in [0.0, 0.05, 0.1, 0.2, 0.3]:
    乱数 = np.random.default_rng(0)
    X2 = Xte.copy()
    X2["hours-per-week"] = X2["hours-per-week"] * (1 + 乱数.normal(0, 割合, len(X2)))
    名 = "そのまま" if 割合 == 0 else f"±{割合*100:.0f}% のばらつき"
    print(f"{名:<22}{モデル.score(X2, yte)*100:>9.1f}%")

print()
print("崩れましたか? 私の予想は「崩れる」でしたが、外れました。")
print()
print("【比較】第6回では、画像を4画素ずらすと 95.1% → 45.1% に落ちました。")
print("        でも今回の表形式データでは、ばらつきに強いようです。")
print()
print("→「毎回崩れる」わけではありません。")
print("  ただし、測らなければ「大丈夫」とも言えませんでした。")

## 視点4(第13回)自信は較正されているか

モデルが「90%の確信がある」と言うとき、実際に90%正しいのでしょうか。

第13回の GPT-2 では、**存在しない国の首都にも実在の国と同じくらいの自信**を示しました。
このモデルはどうでしょうか。

In [ ]:
# ===== 視点4(第13回)自信は較正されているか =====
print(f"{'モデルの自信':<14}{'件数':>8}{'実際に高収入だった率':>22}")
print("-" * 46)
点, 実 = [], []
for lo, hi in [(0.5,0.6), (0.6,0.7), (0.7,0.8), (0.8,0.9), (0.9,1.01)]:
    g = (確率 >= lo) & (確率 < hi)
    if g.sum() < 20: continue
    点.append((lo+hi)/2*100); 実.append(yte[g].mean()*100)
    print(f"{lo:.0%}〜{hi:.0%}{'':<6}{g.sum():>8,}{yte[g].mean()*100:>21.1f}%")

plt.figure(figsize=(5.5, 5))
plt.plot([40,100],[40,100], "--", color="gray", label="完全に較正されている線")
plt.plot(点, 実, "o-", ms=9, lw=2, color="#3A46A8", label="このモデル")
plt.xlabel("モデルの自信(%)"); plt.ylabel("実際に高収入だった率(%)")
plt.title("自信と実際は、合っているか"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("点が破線の近くにあれば、自信は信用できます。")
print("第13回の GPT-2 では、実在の国と架空の国で自信がほとんど同じでした。")
print("→ モデルによって、まったく違います。だから測るしかありません。")

## 視点5(第14回)属性ごとに見るとどうか

全体の数字が良くても、**集団ごとに分けると違う絵**が見えることがあります。

ここで、**正解率だけを見ると誤解する**という現象が起きます。よく見てください。

In [ ]:
# ===== 視点5(第14回)属性ごとに見るとどうか =====
print(f"{'':<8}{'正解率':>10}{'再現率':>10}{'高収入と判定された率':>22}")
print("-" * 52)
for 名, 印 in [("男性", 1), ("女性", 0)]:
    g = 性te == 印
    q, t2 = 予測[g], yte[g]
    tp2 = ((q==1)&(t2==1)).sum(); fn2 = ((q==0)&(t2==1)).sum()
    print(f"{名:<8}{(q==t2).mean()*100:>9.1f}%{tp2/max(tp2+fn2,1)*100:>9.1f}%{q.mean()*100:>21.1f}%")

print()
print("【よく見てください】")
print("正解率だけを見ると、女性のほうが高くなっています。")
print("「女性に対して精度が高い」と報告できてしまいます。")
print()
print("でも、高収入と判定された割合は3分の1程度です。同じモデルの、同じ結果です。")
print()
print("なぜか ── 女性のほとんどが「低収入」と判定され、実際そのとおりだったからです。")
print("多数派を当てれば、正解率は上がります(第3回)。")
print("→「正解率が高い」ことは、その集団がうまく扱われている証拠にはなりません。")

## まとめ — 1つの数字は、5つに分かれた

In [ ]:
# ===== まとめ:1つの数字は、5つに分かれた =====
print("■ 見出しの数字:正解率 {:.1f}%".format((予測==yte).mean()*100))
print()
print("【問題が見つかった視点】")
print(f"  視点1 再現率 {((予測==1)&(yte==1)).sum()/((yte==1).sum())*100:.1f}% ── 4割近くを見逃している")
print(f"        単純なモデルとの差は {(予測==yte).mean()*100-(1-yte.mean())*100:.1f} ポイントだけ")
print("  視点2 深くすると訓練だけ上がり、テストは下がる")
print(f"  視点5 男女で合格率が {予測[性te==1].mean()*100:.1f}% vs {予測[性te==0].mean()*100:.1f}%")
print()
print("【問題が見つからなかった視点】")
print("  視点3 入力のばらつきに強かった")
print("  視点4 自信はよく較正されていた")
print()
print("=" * 58)
print("84.6% という1つの数字は、少なくとも5つの数字に分かれました。")
print("そのうち3つは、その数字を見なければ気づけません。")
print()
print("そして ──「毎回問題が出る」わけではありませんでした。")
print("でも、測らなければ「問題ない」とも言えませんでした。")
print("=" * 58)

> **今日いちばん持ち帰ってほしいこと**
>
> 5つの視点のうち、**3つで問題が見つかり、2つでは見つかりませんでした。**
>
> この授業は毎回「実は問題があります」という話をしてきましたが、
> **「毎回問題が出る」わけではありません。**
>
> でも、**測らなければ「問題ない」とも言えません。**
> それが、この授業でいちばん伝えたかったことです。

## 自分で試す(最終課題用)

In [ ]:
# ===== 自分で試す(最終課題用)=====
# (a) しきい値を変えて、見逃しと空振りを動かす
print(f"{'しきい値':>8}{'正解率':>10}{'適合率':>10}{'再現率':>10}{'高収入と判定':>14}")
print("-" * 54)
for し in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = (確率 >= し).astype(int)
    tp = ((p==1)&(yte==1)).sum(); fp = ((p==1)&(yte==0)).sum(); fn = ((p==0)&(yte==1)).sum()
    適 = tp/max(tp+fp,1)*100; 再 = tp/max(tp+fn,1)*100
    print(f"{し:>8.1f}{(p==yte).mean()*100:>9.1f}%{適:>9.1f}%{再:>9.1f}%{p.mean()*100:>13.1f}%")

print()
print("どの用途なら、どのしきい値を選びますか。理由も考えてください。")
print("  ・見逃しが致命的な用途(病気の検出、不正の検知)→ 低いしきい値")
print("  ・誤って人手をかけると高くつく用途(自動承認)→ 高いしきい値")
print()
print("「どちらの誤りが痛いか」は技術では決まりません。")
print("用途を知っている人が決めることです。")

## この授業の持ち帰り ── 14の質問

技術の名前は忘れても、これだけは残してください。

| # | 質問 | 出どころ |
|---|---|---|
| 1 | もっともらしさで足りる仕事か、正確さが要る仕事か | 第1回 トークン・確率 |
| 2 | その精度は、学習に使っていないデータでの数字か | 第2回 訓練と検証の分離 |
| 3 | 見逃しと空振り、それぞれ何件か | 第3回 適合率・再現率 |
| 4 | その改善は、非線形を足したからか | 第4回 層とReLU |
| 5 | 訓練誤差だけでなく、検証誤差はどうだったか | 第5回 過学習 |
| 6 | それは、どういう変化まで耐えられるか | 第6回 頑健性 |
| 7 | 意味を理解しているのか、似た文脈を測っているのか | 第7回 埋め込み |
| 8 | その図は、どのモデルの、どの層の、どのヘッドか | 第8回 注意の可視化 |
| 9 | その数字が上がったとき、実際の振る舞いはどうだったか | 第9回 報酬設計 |
| 10 | その改善は、私の用途でも改善か | 第10回 ベンチマークと用途 |
| 11 | その出典、実際に開いて読んだか | 第11回 RAG |
| 12 | その数字は、何と比べた数字か | 第12回 比較対象 |
| 13 | それ、言い方を変えて3回聞いたか | 第13回 自己一貫性 |
| 14 | その説明は誰に向けたものか / どの公平を選び、何を諦めたか | 第14回 説明可能性・公平性 |

> **注意**:14の質問は、いつでも全部使うものではありません。
> たとえば 1・7・8・11・13 は言語モデルについての質問なので、今回のような表形式データの分類器には当てはまりません。
>
> **「どれが当てはまるか」を選ぶこと自体が、技術を理解しているということです。**

## 最終課題(manaba)

**締切:講義最終週の日曜 23:59**

### 課題A:身のまわりの場面を検討する
自分の身のまわりで、AI が使われている(または使えそうな)場面を**1つ**選びなさい。
その場面に対して、**14の質問のうち当てはまるものを5つ**選び、それぞれについて書きなさい。

- なぜその質問が当てはまるのか
- その質問に答えるには、**何を測ればよいのか**
- 測った結果が**どうなら、使ってよいと判断するのか**

> 場面の例:大学の成績評価 / バイト先のシフト作成 / 就職活動の書類選考 /
> SNSのおすすめ / 翻訳 / 医療画像の診断支援 …

### 課題B:考えが変わったことを書く
この授業で、いちばん**「思っていたのと違った」**ことを1つ挙げ、**300字以上**で書きなさい。

- 受講前は、どう思っていたか
- **どの回の、どの数字で**、それが変わったか(**具体的な数字を挙げること**)
- いまは、どう考えているか

> 「勉強になりました」式の感想文は評価しません。**数字を1つ以上、必ず引用してください。**

---

### 評価基準(先に開示します)

| 観点 | 見ること | 落とし穴 |
|---|---|---|
| 質問の選びかた | その場面に本当に当てはまる質問を選べているか | 当てはまらない質問を並べていないか |
| 測りかたの具体性 | 「何を測るか」が実行可能な形で書けているか | 「きちんと評価する」では不可 |
| 判断の基準 | どうなら使ってよいと言えるか | 基準が後付けになっていないか |
| 数字の引用 | 授業の実測値を正しく引用できているか | 数字なしの感想文は評価しません |
| **限界の自覚** | **自分の検討では分からないことを書けているか** | **「これで十分」と書いていないか** |

**いちばん評価するのは、5つ目です。**

分かったことを書くのは、誰でもできます。
難しいのは、**分からなかったことを、分からないと書くこと**です。
それができる人は、間違えたときに気づけます。

---

**提出**:PDF 1本(A4・3〜5ページ目安)。ノートブックの図を貼ってもかまいません。
分量よりも、**根拠として数字を挙げられているか**を見ます。

---

15回、おつかれさまでした。
このノートブックも、他の14回ぶんも、いつでも開いて動かせます。手元に置いておいてください。